In [1]:
# ===== CELL 0: installs =====
!pip install -q ultralytics boxmot
!pip install -q --no-deps motmetrics xmltodict
!pip install -q "numpy<2.1"          # boxmot numpy 2.5 laata hai, scipy usse crash karta hai
print("INSTALL DONE — ab kernel restart karo (neeche padho)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 10.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 6.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, whi

In [2]:
# ===== CELL 0b: verify (restart ke baad) =====
import numpy as np
if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)
    print("numpy shim lagaya")

import ultralytics, boxmot, motmetrics, torch, scipy
print("numpy:", np.__version__, "| scipy:", scipy.__version__)
print("ultralytics:", ultralytics.__version__, "| boxmot:", boxmot.__version__,
      "| motmetrics:", motmetrics.__version__)
print("cuda:", torch.cuda.is_available())

numpy shim lagaya
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
numpy: 2.0.2 | scipy: 1.16.3
ultralytics: 8.4.154 | boxmot: 25.0.0 | motmetrics: 1.4.0
cuda: True


In [3]:
# ===== CELL 1: inputs =====
from pathlib import Path

vids = sorted(Path("/kaggle/input").rglob("*.mp4"))
for v in vids:
    print("video:", v.name, f"| {v.stat().st_size/1e6:.0f} MB")

DEV_VIDEO  = [v for v in vids if "chunk_010" in v.name][0]
TEST_VIDEO = [v for v in vids if "chunk_032" in v.name][0]

GT_JSON = next(p for p in Path("/kaggle/input").rglob("*.json")
               if "video_tracking" in p.name)
print("GT :", GT_JSON.name)

# trained Re-ID weights (naya input)
w = list(Path("/kaggle/input").rglob("osnet_x1_0_cam12.pt"))
assert w, "osnet_x1_0_cam12.pt nahi mila -> trained-weights dataset attach karo"
TRAINED_REID = w[0]
print("ReID weights:", TRAINED_REID, f"| {TRAINED_REID.stat().st_size/1e6:.1f} MB")

DEV_KEY, TEST_KEY = DEV_VIDEO.stem, TEST_VIDEO.stem
print("DEV :", DEV_KEY)
print("TEST:", TEST_KEY)

video: Cam_12_20260222T000000Z_20260223T000000Z_10fps_chunk_010.mp4 | 310 MB
video: Cam_12_20260222T000000Z_20260223T000000Z_10fps_chunk_032.mp4 | 77 MB
GT : tracking_db.video_tracking (2).json
ReID weights: /kaggle/input/notebooks/payal1238gupta/phase4-re-id/reid/osnet_x1_0_cam12.pt | 9.4 MB
DEV : Cam_12_20260222T000000Z_20260223T000000Z_10fps_chunk_010
TEST: Cam_12_20260222T000000Z_20260223T000000Z_10fps_chunk_032


In [4]:
# ===== CELL 2: GT =====
import json

db = json.load(open(GT_JSON))

def load_gt(key):
    d = {}
    for e in db:
        if e["video"] != key:
            continue
        f = e["frame_number"] + 1                    # MOT 1-based
        rows = [(t["track_id"], t["bbox"][0], t["bbox"][1],
                 t["bbox"][2] - t["bbox"][0], t["bbox"][3] - t["bbox"][1])
                for t in e.get("tracks", [])]
        if rows:
            d[f] = rows
    return d

gt_dev, gt_test = load_gt(DEV_KEY), load_gt(TEST_KEY)
print("GT frames -> dev:", len(gt_dev), "| test:", len(gt_test))
assert gt_dev and gt_test, "GT khaali hai -> video/GT keys match nahi kar rahe"

GT frames -> dev: 916 | test: 284


In [5]:
# ===== CELL 3: ByteTrack baseline =====
import cv2, time
from ultralytics import YOLO

person_model = YOLO("yolov8n.pt")

def run_bytetrack(video_path, tag, conf=0.25):
    stream = person_model.track(source=str(video_path), conf=conf, classes=[0],
                                tracker="bytetrack.yaml", persist=True,
                                stream=True, verbose=False)
    all_ids, rows, n = set(), [], 0
    t0 = time.time()
    for res in stream:
        if res.boxes.id is not None:
            for box, tid, cf in zip(res.boxes.xyxy, res.boxes.id, res.boxes.conf):
                x1, y1, x2, y2 = map(int, box.tolist())
                tid = int(tid); all_ids.add(tid)
                rows.append([n, tid, x1, y1, x2 - x1, y2 - y1, float(cf)])
        n += 1
        if n % 4000 == 0:
            print(f"  {tag}: {n} frames | {n/(time.time()-t0):.1f} fps | ids={len(all_ids)}")
    with open(f"tracks_base_{tag}.txt", "w") as f:
        for fr, tid, x, y, w_, h_, cf in rows:
            f.write(f"{fr+1},{tid},{x},{y},{w_},{h_},{cf:.3f},-1,-1,-1\n")
    print(f"{tag}: {n} frames | unique ids = {len(all_ids)} | {round((time.time()-t0)/60,1)} min")
    return len(all_ids)

uid_base_dev  = run_bytetrack(DEV_VIDEO,  "dev")
uid_base_test = run_bytetrack(TEST_VIDEO, "test")

  dev: 4000 frames | 83.1 fps | ids=20
  dev: 8000 frames | 90.4 fps | ids=44
  dev: 12000 frames | 93.1 fps | ids=69
  dev: 16000 frames | 94.5 fps | ids=96
  dev: 20000 frames | 95.1 fps | ids=131
  dev: 24000 frames | 95.6 fps | ids=159
  dev: 28000 frames | 95.9 fps | ids=179
dev: 28365 frames | unique ids = 180 | 4.9 min
  test: 4000 frames | 115.1 fps | ids=10
test: 6599 frames | unique ids = 14 | 1.0 min


In [6]:
# ===== CELL 4: mask + eval =====
import numpy as np
import motmetrics as mm

def load_pred(path):
    d = {}
    for line in open(path):
        p = line.strip().split(",")
        f, tid = int(p[0]), int(p[1])
        d.setdefault(f, []).append((tid, *map(float, p[2:6])))
    return d

pred_base_dev  = load_pred("tracks_base_dev.txt")
pred_base_test = load_pred("tracks_base_test.txt")

# poster box (baseline DEV preds se, wahi cx/cy window)
cand = []
for f, boxes in pred_base_dev.items():
    for tid, x, y, w, h in boxes:
        cx, cy = x + w/2, y + h/2
        if 450 <= cx <= 600 and 130 <= cy <= 280:
            cand.append([x, y, x + w, y + h])
cand = np.array(cand)
print("poster candidate boxes:", len(cand))
POSTER_BOX = [float(np.percentile(cand[:, 0], 5)), float(np.percentile(cand[:, 1], 5)),
              float(np.percentile(cand[:, 2], 95)), float(np.percentile(cand[:, 3], 95))]
MASK_THR = 0.95
print("POSTER_BOX =", [round(v) for v in POSTER_BOX], "| thr =", MASK_THR)

def in_poster(x, y, w, h, box=POSTER_BOX, thr=MASK_THR):
    px1, py1, px2, py2 = box
    ix1, iy1 = max(x, px1), max(y, py1)
    ix2, iy2 = min(x + w, px2), min(y + h, py2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    return inter / max(w * h, 1) > thr

def clean(pred):
    o = {f: [b for b in bs if not in_poster(*b[1:])] for f, bs in pred.items()}
    return {f: v for f, v in o.items() if v}

gt_lost = sum(1 for f in gt_dev for (tid, x, y, w, h) in gt_dev[f] if in_poster(x, y, w, h))
print(f"GT boxes mask ke andar: {gt_lost} ({100*gt_lost/sum(len(v) for v in gt_dev.values()):.1f}%)")

def evaluate(gt, pred, name):
    acc = mm.MOTAccumulator(auto_id=False)
    for f in sorted(gt):
        g, p = gt[f], pred.get(f, [])
        gb = np.array([r[1:] for r in g], dtype=float).reshape(-1, 4)
        pb = np.array([r[1:] for r in p], dtype=float).reshape(-1, 4)
        dist = mm.distances.iou_matrix(gb, pb, max_iou=0.5)
        acc.update([r[0] for r in g], [r[0] for r in p], dist, frameid=f)
    mh = mm.metrics.create()
    return mh.compute(acc, name=name, metrics=[
        "num_frames", "mota", "motp", "idf1", "num_switches", "num_fragmentations",
        "num_false_positives", "num_misses", "mostly_tracked", "mostly_lost"])

print("eval ready")

poster candidate boxes: 29759
POSTER_BOX = [385, 84, 622, 381] | thr = 0.95
GT boxes mask ke andar: 59 (5.5%)
eval ready


In [7]:
# ===== CELL 6: BotSort runs (BoxMOT 25 API) =====
import numpy as np, time, torch
from pathlib import Path
from boxmot import BotSort

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

def to_dets(boxes):
    if len(boxes) == 0:
        return np.empty((0, 6))
    xyxy = boxes.xyxy.cpu().numpy()
    conf = boxes.conf.cpu().numpy().reshape(-1, 1)
    cls = boxes.cls.cpu().numpy().reshape(-1, 1)
    return np.concatenate([xyxy, conf, cls], axis=1)

def run_botsort(reid_weights, video, out_name):
    #tr = BotSort(reid_weights=Path(reid_weights), device=DEVICE, half=False, track_buffer=60)
    tr = BotSort(reid_weights=Path(reid_weights), device=DEVICE, half=False, track_buffer=60,
             proximity_thresh=0.9, appearance_thresh=0.5)
    stream = person_model.predict(source=str(video), conf=0.25, classes=[0], stream=True, verbose=False)
    rows, ids, n = [], set(), 0
    t0 = time.time()
    for res in stream:
        out = np.array(tr.update(to_dets(res.boxes), res.orig_img))
        if out.size:
            for r in out:
                x1, y1, x2, y2, tid = r[0], r[1], r[2], r[3], int(r[4])
                conf = r[5] if len(r) > 5 else 1.0
                ids.add(tid)
                rows.append([n, tid, x1, y1, x2 - x1, y2 - y1, conf])
        n += 1
        if n % 5000 == 0:
            print("  ", out_name, n, "frames |", round(n / (time.time() - t0), 1), "fps | ids", len(ids))
    with open(out_name, "w") as f:
        for fr, tid, x, y, w_, h_, cf in rows:
            f.write("%d,%d,%.1f,%.1f,%.1f,%.1f,%.3f,-1,-1,-1\n" % (fr + 1, tid, x, y, w_, h_, cf))
    mins = round((time.time() - t0) / 60, 1)
    print(out_name, "|", n, "frames |", len(rows), "rows | unique ids", len(ids), "|", mins, "min")
    return len(ids)

uid_pre_dev  = run_botsort("osnet_x0_25_msmt17.pt", DEV_VIDEO,  "tracks_pre_dev.txt")
uid_pre_test = run_botsort("osnet_x0_25_msmt17.pt", TEST_VIDEO, "tracks_pre_test.txt")
uid_own_dev  = run_botsort(TRAINED_REID,            DEV_VIDEO,  "tracks_own_dev.txt")
uid_own_test = run_botsort(TRAINED_REID,            TEST_VIDEO, "tracks_own_test.txt")

device: cuda:0


INFO     BotSort: det_thresh=0.3, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False,          
         class_ids=None, asso_func=iou, variable_dt=False

INFO     BoxMOT v25.0.0 🚀 Python-3.12.13 torch-2.10.0+cu128                                                       
         CUDA:0 (Tesla T4, 14912MiB)

INFO     /usr/local/lib/python3.12/dist-packages/models/osnet_x0_25_msmt17.pt

INFO     [PID 24] Downloading ReID weights from https://drive.google.com/uc?id=1sSwXSUlj4_tHZequ_iZ8w_Jh0VaRQMqF → 
         /usr/local/lib/python3.12/dist-packages/models/osnet_x0_25_msmt17.pt

INFO     Downloading osnet_x0_25_msmt17.pt...

... (19 outputs left out)

INFO     [PID 24] Found existing ReID weights at                                                                   
         /kaggle/input/notebooks/payal1238gupta/phase4-re-id/reid/osnet_x1_0_cam12.pt; skipping download.

INFO     Loaded deployed weights from /kaggle/input/notebooks/payal1238gupta/phase4-re-id/reid/osnet_x1_0_cam12.pt:
         565/565 required tensors, 100.00% required elements

WARNING  ECC correlation 0.439 is below 0.500; returning identity warp.

   tracks_own_test.txt 5000 frames | 70.5 fps | ids 8
tracks_own_test.txt | 6599 frames | 279 rows | unique ids 9 | 1.5 min


In [8]:
# ===== CELL 7: parse =====
pred_pre_dev  = load_pred("tracks_pre_dev.txt")
pred_pre_test = load_pred("tracks_pre_test.txt")
pred_own_dev  = load_pred("tracks_own_dev.txt")
pred_own_test = load_pred("tracks_own_test.txt")

print("full-video unique IDs:")
print("  DEV :", "ByteTrack", uid_base_dev, "| pretrained", uid_pre_dev, "| trained", uid_own_dev)
print("  TEST:", "ByteTrack", uid_base_test, "| pretrained", uid_pre_test, "| trained", uid_own_test)
uids = lambda pred: len({t for v in pred.values() for (t, *_) in v})

full-video unique IDs:
  DEV : ByteTrack 180 | pretrained 12 | trained 13
  TEST: ByteTrack 14 | pretrained 9 | trained 9


In [9]:
# ===== CELL 8: final table =====
import pandas as pd, json
import motmetrics as mm

full = pd.concat([
    evaluate(gt_dev,  clean(pred_base_dev),  "DEV  | ByteTrack"),
    evaluate(gt_dev,  clean(pred_pre_dev),   "DEV  | BotSORT+OSNet pretrained"),
    evaluate(gt_dev,  clean(pred_own_dev),   "DEV  | BotSORT+OSNet TRAINED"),
    evaluate(gt_test, clean(pred_base_test), "TEST | ByteTrack"),
    evaluate(gt_test, clean(pred_pre_test),  "TEST | BotSORT+OSNet pretrained"),
    evaluate(gt_test, clean(pred_own_test),  "TEST | BotSORT+OSNet TRAINED"),
])
print(mm.io.render_summary(full, formatters=mm.metrics.create().formatters,
                           namemap=mm.io.motchallenge_metric_names))
full.to_csv("phase4_final_comparison.csv")

json.dump({
    "unique_ids_full_video": {
        "dev":  {"bytetrack": uid_base_dev,  "pretrained": uids(pred_pre_dev),  "trained": uids(pred_own_dev)},
        "test": {"bytetrack": uid_base_test, "pretrained": uids(pred_pre_test), "trained": uids(pred_own_test)},
    },
    "mask": {"poster_box": [round(v, 1) for v in POSTER_BOX], "thr": MASK_THR},
    "annotated_frames": {"dev": len(gt_dev), "test": len(gt_test)},
}, open("phase4_comparison_summary.json", "w"), indent=2)
print("\nsaved: phase4_final_comparison.csv | phase4_comparison_summary.json")

                                num_frames  MOTA  MOTP  IDF1 IDs  FM  FP  FN MT ML
DEV  | ByteTrack                       916 58.5% 0.212 60.1%   9  39 119 315  5  3
DEV  | BotSORT+OSNet pretrained        916 66.3% 0.216 62.3%   7  33  12 341  6  3
DEV  | BotSORT+OSNet TRAINED           916 66.3% 0.216 62.3%   7  33  12 341  6  3
TEST | ByteTrack                       284 72.9% 0.232 78.4%   3   7  17  79  7  2
TEST | BotSORT+OSNet pretrained        284 71.5% 0.232 67.9%   3   5   5  96  6  2
TEST | BotSORT+OSNet TRAINED           284 71.5% 0.232 67.9%   3   5   5  96  6  2

saved: phase4_final_comparison.csv | phase4_comparison_summary.json
